# Milestone 2: Deep Knowledge Tracing Training & Evaluation

This notebook trains and evaluates Deep Knowledge Tracing architectures (**DKT**, **AKT**, **SAKT**) using `pykt-toolkit` on GPU.

### Key Guidelines:
1. **Validation Model Selection**: Checkpoints are chosen based strictly on **Validation AUC** (fold 1).
2. **Held-Out Test Evaluation**: Held-out test learners (fold -1 / fold 2) are evaluated strictly once in the final step.
3. **Baselines**: Includes non-neural baselines (*Constant Global Mean* and *Per-KC Mean*) for comparison.

In [2]:
# Step 0: GPU & Environment Verification
import os
from pathlib import Path
import torch

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device Name: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: Running on CPU. Training will be slower!')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/thesis')
    os.chdir(PROJECT_ROOT)
    print(f'Working directory set to: {os.getcwd()}')
except ImportError:
    PROJECT_ROOT = Path('.').resolve()
    print(f'Running locally. Working directory: {PROJECT_ROOT}')

%pip install -q pykt-toolkit scikit-learn pandas numpy

PyTorch Version: 2.14.1+cu130
CUDA Available : True
GPU Device Name: Tesla T4
Mounted at /content/drive
Working directory set to: /content/drive/MyDrive/thesis


## 1. Setup Common Utilities & Model Configurations

Defines model parameters (`model_configs.json`) and shared helper functions (`kt_common.py` equivalent).

In [3]:
import json
import random
import types
import numpy as np
import pandas as pd

# Guard pyKT turtle import if missing
try:
    import turtle
except Exception:
    import sys
    stub = types.ModuleType('turtle')
    stub.forward = lambda *a, **k: None
    sys.modules['turtle'] = stub

CONFIG_DIR = PROJECT_ROOT / 'knowledge_tracing/configs'
ARTIFACTS_DIR = PROJECT_ROOT / 'knowledge_tracing/artifacts'
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = 'english_kt'
EMB_TYPE = 'qid'
TRAIN_FOLD, VALID_FOLD, TEST_FOLD = 0, 1, -1

# Write model_configs.json
MODEL_CONFIGS = {
    '_note': 'Initial modest hyperparameter configuration',
    'dkt': {
        'model_config': {'emb_size': 128, 'dropout': 0.2},
        'training': {'learning_rate': 0.001, 'batch_size': 64, 'num_epochs': 50}
    },
    'sakt': {
        'model_config': {'seq_len': 200, 'emb_size': 64, 'num_attn_heads': 4, 'dropout': 0.2, 'num_en': 2},
        'training': {'learning_rate': 0.001, 'batch_size': 64, 'num_epochs': 50}
    },
    'akt': {
        'model_config': {'d_model': 64, 'n_blocks': 2, 'dropout': 0.2, 'd_ff': 128, 'kq_same': 1,
                         'final_fc_dim': 256, 'num_attn_heads': 4, 'separate_qa': False, 'l2': 1e-5},
        'training': {'learning_rate': 0.0005, 'batch_size': 64, 'num_epochs': 50}
    }
}
(CONFIG_DIR / 'model_configs.json').write_text(json.dumps(MODEL_CONFIGS, indent=2))

def load_data_config():
    cfg_path = CONFIG_DIR / 'data_config.json'
    if not cfg_path.exists():
        raise FileNotFoundError(f'{cfg_path} not found - run notebook 01 first.')
    cfg = json.loads(cfg_path.read_text())[DATASET_NAME]
    cfg['dpath'] = str((PROJECT_ROOT / cfg['dpath']).resolve())
    return cfg

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def apply_cpu_shim():
    if torch.cuda.is_available():
        return False
    import pykt.datasets.data_loader as dl
    dl.LongTensor = torch.LongTensor
    dl.FloatTensor = torch.FloatTensor
    return True

def make_dataset(split: str, data_cfg: dict):
    from pykt.datasets.data_loader import KTDataset
    apply_cpu_shim()
    dpath = Path(data_cfg['dpath'])
    if split == 'train':
        return KTDataset(str(dpath / data_cfg['train_valid_file']), data_cfg['input_type'], {TRAIN_FOLD})
    if split == 'valid':
        return KTDataset(str(dpath / data_cfg['train_valid_file']), data_cfg['input_type'], {VALID_FOLD})
    if split == 'test':
        return KTDataset(str(dpath / data_cfg['test_file']), data_cfg['input_type'], {TEST_FOLD})
    raise ValueError(f'Unknown split: {split}')

print('Shared functions and model configurations loaded successfully.')

Shared functions and model configurations loaded successfully.


## 2. Train Deep KT Models (`train.py` Logic)

Train DKT, SAKT, and AKT on fold 0 and evaluate validation performance on fold 1.

In [4]:
import contextlib
import shutil
import sys
import time
from importlib import metadata
from torch.utils.data import DataLoader
from pykt.models import init_model, train_model

class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, text):
        for stream in self.streams:
            stream.write(text)

    def flush(self):
        for stream in self.streams:
            stream.flush()

# 07-11-2026: def train_one_model(model_name: str, seed: int = 42, epochs_override: int = None):
#     data_cfg = load_data_config()
#     model_cfg = MODEL_CONFIGS[model_name]['model_config']
#     training_cfg = dict(MODEL_CONFIGS[model_name]['training'])
#     if epochs_override:
#         training_cfg['num_epochs'] = epochs_override

#     run_dir = ARTIFACTS_DIR / model_name / f'seed{seed}'
def train_one_model(model_name: str, seed: int = 42, epochs_override: int = None,
                    model_cfg_override: dict = None, training_cfg_override: dict = None,
                    run_dir=None):
    data_cfg = load_data_config()
    model_cfg = {**MODEL_CONFIGS[model_name]['model_config'], **(model_cfg_override or {})}
    training_cfg = {**MODEL_CONFIGS[model_name]['training'], **(training_cfg_override or {})}
    if epochs_override:
        training_cfg['num_epochs'] = epochs_override

    run_dir = Path(run_dir) if run_dir else ARTIFACTS_DIR / model_name / f'seed{seed}'
    run_dir.mkdir(parents=True, exist_ok=True)

    # Resume support: skip a run that already finished with the SAME configuration
    summary_path = run_dir / 'run_summary.json'
    if summary_path.exists():
        saved = json.loads(summary_path.read_text())
        if saved['model_config'] == model_cfg and saved['training'] == training_cfg:
            print(f'Skipping {model_name} seed {seed}: already finished ({summary_path})')
            return saved

    with open(run_dir / 'train.log', 'w') as log_file, contextlib.redirect_stdout(Tee(sys.stdout, log_file)):
        print(f'=== {model_name.upper()} | seed {seed} | {training_cfg} ===')
        set_seed(seed)

        train_ds = make_dataset('train', data_cfg)
        valid_ds = make_dataset('valid', data_cfg)
        generator = torch.Generator().manual_seed(seed)
        train_loader = DataLoader(train_ds, batch_size=training_cfg['batch_size'], shuffle=True, generator=generator)
        valid_loader = DataLoader(valid_ds, batch_size=training_cfg['batch_size'], shuffle=False)
        print(f'Train learners: {len(train_ds)} | Valid learners: {len(valid_ds)}')

        model = init_model(model_name, model_cfg, data_cfg, EMB_TYPE)
        parameter_count = sum(parameter.numel() for parameter in model.parameters())
        # optimizer = torch.optim.Adam(model.parameters(), lr=training_cfg['learning_rate'])
        optimizer = torch.optim.Adam(model.parameters(), lr=training_cfg['learning_rate'], weight_decay=training_cfg.get('weight_decay', 0.0))

        started_at = time.time()
        # No test loader is passed; checkpoint selection uses the validation fold only.
        result = train_model(
            model, train_loader, valid_loader, training_cfg['num_epochs'], optimizer,
            str(run_dir), None, None, True
        )
        valid_auc, valid_acc, best_epoch = result[4], result[5], result[6]
        elapsed_seconds = time.time() - started_at

        checkpoint = run_dir / f'{EMB_TYPE}_model.ckpt'
        if not checkpoint.exists():
            raise RuntimeError(f'pyKT did not write {checkpoint}')
        shutil.copy(checkpoint, run_dir / 'best_model.pth')

        summary = {
            'model': model_name,
            'seed': seed,
            'best_epoch': int(best_epoch),
            'valid_auc': float(valid_auc),
            'valid_acc': float(valid_acc),
            'n_params': int(parameter_count),
            'train_seconds': round(elapsed_seconds, 1),
            'model_config': model_cfg,
            'training': training_cfg,
            'emb_type': EMB_TYPE,
            'pykt_toolkit_version': metadata.version('pykt-toolkit'),
            'torch_version': torch.__version__
        }
        (run_dir / 'run_summary.json').write_text(json.dumps(summary, indent=2))
        print(
            f'Finished {model_name}: best epoch {best_epoch}, validation AUC {valid_auc:.4f}, '
            f'validation accuracy {valid_acc:.4f}, {elapsed_seconds:.1f}s -> {run_dir}'
        )
    return summary

MODELS_TO_TRAIN = ['dkt', 'akt', 'sakt']
# SEEDS = [42]
SEEDS = [42, 43, 44]
training_summaries = [
    train_one_model(model_name, seed=seed)
    for model_name in MODELS_TO_TRAIN
    for seed in SEEDS
]

print('\n' + '=' * 60 + '\nTRAINING SUMMARY (validation fold only)\n' + '=' * 60)
for summary in training_summaries:
    print(
        f"{summary['model'].upper():<5} seed {summary['seed']:<4} "
        f"best epoch {summary['best_epoch']:<3} valid AUC {summary['valid_auc']:.4f} "
        f"valid ACC {summary['valid_acc']:.4f} params {summary['n_params']:,} "
        f"{summary['train_seconds']}s"
    )

Skipping dkt seed 42: already finished (/content/drive/MyDrive/thesis/knowledge_tracing/artifacts/dkt/seed42/run_summary.json)
Skipping dkt seed 43: already finished (/content/drive/MyDrive/thesis/knowledge_tracing/artifacts/dkt/seed43/run_summary.json)
Skipping dkt seed 44: already finished (/content/drive/MyDrive/thesis/knowledge_tracing/artifacts/dkt/seed44/run_summary.json)
Skipping akt seed 42: already finished (/content/drive/MyDrive/thesis/knowledge_tracing/artifacts/akt/seed42/run_summary.json)
=== AKT | seed 43 | {'learning_rate': 0.0005, 'batch_size': 64, 'num_epochs': 50} ===
Start preprocessing /content/drive/MyDrive/thesis/data/processed/pykt_english/train_valid_sequences.csv fold: _0...


/usr/local/lib/python3.13/dist-packages/pykt/datasets/data_loader.py:152: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  dori[key] = LongTensor(dori[key])


interaction_num: 52521
file path: /content/drive/MyDrive/thesis/data/processed/pykt_english/train_valid_sequences.csv, qlen: 840, clen: 840, rlen: 840
Start preprocessing /content/drive/MyDrive/thesis/data/processed/pykt_english/train_valid_sequences.csv fold: _1...
interaction_num: 7522
file path: /content/drive/MyDrive/thesis/data/processed/pykt_english/train_valid_sequences.csv, qlen: 120, clen: 120, rlen: 120
Train learners: 840 | Valid learners: 120
ts.shape: (7402,), ps.shape: (7402,)
Epoch: 1, validauc: 0.8091, validacc: 0.6606, best epoch: 1, best auc: 0.8091, train loss: 0.6081843828224177, emb_type: qid, model: akt, save_dir: /content/drive/MyDrive/thesis/knowledge_tracing/artifacts/akt/seed43
            testauc: -1, testacc: -1, window_testauc: -1, window_testacc: -1
ts.shape: (7402,), ps.shape: (7402,)
Epoch: 2, validauc: 0.8158, validacc: 0.7522, best epoch: 2, best auc: 0.8158, train loss: 0.517515465004694, emb_type: qid, model: akt, save_dir: /content/drive/MyDrive/the

In [5]:
# ===== Small random hyperparameter search (selection by VALIDATION AUC only) =====
# Run this BETWEEN the training cell and the evaluation cell. It never touches the test split.
import random as pyrandom

N_TRIALS = 8          # trials per model; trial 0 is always the current default configuration
SEARCH_SEED = 0

SEARCH_SPACE = {
    'dkt': {
        'model': {'emb_size': [64, 128, 256], 'dropout': [0.1, 0.2, 0.3, 0.5]},
        'train': {'learning_rate': [2e-3, 1e-3, 5e-4], 'batch_size': [32, 64], 'weight_decay': [0.0, 1e-5, 1e-4]},
    },
    'sakt': {
        'model': {'emb_size': [64, 128], 'num_attn_heads': [2, 4, 8], 'num_en': [1, 2, 3], 'dropout': [0.1, 0.2, 0.3, 0.4]},
        'train': {'learning_rate': [2e-3, 1e-3, 5e-4], 'batch_size': [32, 64], 'weight_decay': [0.0, 1e-5, 1e-4]},
    },
    'akt': {
        'model': {'d_model': [64, 128], 'n_blocks': [1, 2, 4], 'd_ff': [128, 256],
                  'dropout': [0.1, 0.2, 0.3], 'num_attn_heads': [4, 8]},
        'train': {'learning_rate': [1e-3, 5e-4, 2.5e-4], 'batch_size': [32, 64], 'weight_decay': [0.0, 1e-5]},
    },
}

def sample_trials(model_name, n_trials, rng):
    space, seen = SEARCH_SPACE[model_name], {json.dumps([{}, {}])}
    trials = [({}, {})]                                   # trial 0 = defaults from MODEL_CONFIGS
    while len(trials) < n_trials:
        m = {k: rng.choice(v) for k, v in space['model'].items()}
        t = {k: rng.choice(v) for k, v in space['train'].items()}
        key = json.dumps([m, t], sort_keys=True)
        if key not in seen:
            seen.add(key)
            trials.append((m, t))
    return trials

tuning_rows = []
for model_name in MODELS_TO_TRAIN:
    rng = pyrandom.Random(SEARCH_SEED)
    for i, (m_over, t_over) in enumerate(sample_trials(model_name, N_TRIALS, rng)):
        trial_dir = ARTIFACTS_DIR / 'tuning' / model_name / f'trial{i}'
        s = train_one_model(model_name, seed=42, model_cfg_override=m_over,
                            training_cfg_override=t_over, run_dir=trial_dir)
        for leftover in ('qid_model.ckpt', 'best_model.pth'):   # keep Drive small; logs + summary stay
            (trial_dir / leftover).unlink(missing_ok=True)
        tuning_rows.append({'model': model_name, 'trial': i, 'valid_auc': round(s['valid_auc'], 4),
                            'best_epoch': s['best_epoch'], 'seconds': s['train_seconds'],
                            'model_config': json.dumps(s['model_config']),
                            'training': json.dumps(s['training'])})

tuning_df = pd.DataFrame(tuning_rows)
tuning_df.to_csv(ARTIFACTS_DIR / 'tuning' / 'tuning_results.csv', index=False)
pd.set_option('display.max_colwidth', 200, 'display.width', 250)
for model_name, group in tuning_df.groupby('model', sort=False):
    default_auc = group.loc[group['trial'] == 0, 'valid_auc'].iloc[0]
    print(f'\n=== {model_name.upper()}: default (trial 0) valid AUC = {default_auc} ===')
    print(group.sort_values('valid_auc', ascending=False).head(3)[['trial', 'valid_auc', 'best_epoch', 'model_config', 'training']].to_string(index=False))

=== DKT | seed 42 | {'learning_rate': 0.001, 'batch_size': 64, 'num_epochs': 50} ===
Read data from processed file: /content/drive/MyDrive/thesis/data/processed/pykt_english/train_valid_sequences.csv_0.pkl
file path: /content/drive/MyDrive/thesis/data/processed/pykt_english/train_valid_sequences.csv, qlen: 840, clen: 840, rlen: 840
Read data from processed file: /content/drive/MyDrive/thesis/data/processed/pykt_english/train_valid_sequences.csv_1.pkl
file path: /content/drive/MyDrive/thesis/data/processed/pykt_english/train_valid_sequences.csv, qlen: 120, clen: 120, rlen: 120
Train learners: 840 | Valid learners: 120
ts.shape: (7402,), ps.shape: (7402,)
Epoch: 1, validauc: 0.7329, validacc: 0.7013, best epoch: 1, best auc: 0.7329, train loss: 0.6787620927016996, emb_type: qid, model: dkt, save_dir: /content/drive/MyDrive/thesis/knowledge_tracing/artifacts/tuning/dkt/trial0
            testauc: -1, testacc: -1, window_testauc: -1, window_testacc: -1
ts.shape: (7402,), ps.shape: (7402,)


# 3. Evaluate Models on Held-Out Test Set (`evaluate.py` Logic)

Evaluates trained checkpoints strictly on the held-out **Test** split (fold -1). Calculates **AUC**, **Accuracy**, and **Log Loss**, comparing neural models against non-neural baselines (*Global Constant Mean* and *Per-KC Mean*).

In [12]:
import ast
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from torch.utils.data import DataLoader
from pykt.models import evaluate, init_model

import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

EVAL_DIR = ARTIFACTS_DIR / 'evaluation'
EVAL_DIR.mkdir(parents=True, exist_ok=True)
EPS = 1e-7

def metrics_for(y_true, y_prob):
    y_prob = np.clip(np.asarray(y_prob, dtype=float), EPS, 1 - EPS)
    return {
        'auc': float(roc_auc_score(y_true, y_prob)),
        'accuracy': float(accuracy_score(y_true, (y_prob >= 0.5).astype(int))),
        'log_loss': float(log_loss(y_true, y_prob, labels=[0, 1]))
    }

# def parse_predictions_by_learner(path: Path):
#     ts, ps = [], []
#     for line in path.read_text().splitlines():
#         if not line.strip():
#             continue
#         # per-sequence AUC is the bare token `nan` for one-class learners; unused, so map to None
#         rec = ast.literal_eval(line.replace('nan', 'None'))
#         ts.append(np.asarray(rec[3], dtype=int))
#         ps.append(np.asarray(rec[4], dtype=float))
#     return ts, ps

# def parse_predictions(path: Path):
#     ts, ps = parse_predictions_by_learner(path)
#     y_true, y_prob = np.concatenate(ts), np.concatenate(ps)
#     if np.isnan(y_prob).any():
#         raise RuntimeError(f'NaN probabilities in {path}')
#     return y_true, y_prob

def parse_predictions_by_learner(path: Path):
    ts, ps = [], []

    # Expose 'np' directly to the eval context so strings containing
    # 'np.array' or 'np.nan' can execute correctly.
    eval_namespace = {
        "__builtins__": {},
        "np": np,             # <-- FIX: Expose the numpy module
        "array": np.array,    # Kept in case there are bare 'array(' calls
        "float32": np.float32,
        "float64": np.float64,
        "float": float,
        "int": int,
        "set": set,
        "inf": np.inf,
        "nan": None           # Kept for the bare 'nan' tokens you mentioned
    }

    for line in path.read_text().splitlines():
        if not line.strip():
            continue

        # Now eval() knows what 'np' is when it reads the file
        rec = eval(line, eval_namespace)

        ts.append(np.asarray(rec[3], dtype=int))
        ps.append(np.asarray(rec[4], dtype=float))

    return ts, ps

def parse_predictions(path: Path):
    ts, ps = [], []
    # Restricted execution context for safe evaluation of numpy array reprs
    eval_globals = {
        "__builtins__": None,
        "array": np.array,
        "np": np,
        "nan": np.nan,
        "None": None,
        "true": True,
        "false": False
    }

    for line in path.read_text().splitlines():
        line_str = line.strip()
        if not line_str:
            continue

        try:
            # Attempt standard JSON parse first
            rec = json.loads(line_str.replace('nan', 'null'))
        except json.JSONDecodeError:
            # Fallback to controlled eval for reprs containing array(...)
            rec = eval(line_str, eval_globals)

        ts.extend(rec[3])
        ps.extend(rec[4])

    return np.asarray(ts, dtype=int), np.asarray(ps, dtype=float)

def sequence_values(row, field):
    return [int(value) for value in str(row[field]).split(',')]

def baseline_scores(data_cfg):
    data_dir = Path(data_cfg['dpath'])
    train_valid = pd.read_csv(data_dir / data_cfg['train_valid_file'])
    test = pd.read_csv(data_dir / data_cfg['test_file'])

    kc_sum, kc_count, total_sum, total_count = {}, {}, 0, 0
    for _, row in train_valid[train_valid['fold'] == TRAIN_FOLD].iterrows():
        for concept, response, mask in zip(
            sequence_values(row, 'concepts'),
            sequence_values(row, 'responses'),
            sequence_values(row, 'selectmasks')
        ):
            if mask == -1:
                continue
            kc_sum[concept] = kc_sum.get(concept, 0) + response
            kc_count[concept] = kc_count.get(concept, 0) + 1
            total_sum += response
            total_count += 1
    if total_count == 0:
        raise ValueError('No unpadded training responses available for baselines')
    global_mean = total_sum / total_count

    targets, constant_scores, kc_scores = [], [], []
    for _, row in test.iterrows():
        concepts = sequence_values(row, 'concepts')
        responses = sequence_values(row, 'responses')
        masks = sequence_values(row, 'selectmasks')
        for concept, response, mask in list(zip(concepts, responses, masks))[1:]:
            if mask == -1:
                continue
            targets.append(response)
            constant_scores.append(global_mean)
            kc_scores.append(kc_sum[concept] / kc_count[concept] if concept in kc_count else global_mean)
    targets = np.asarray(targets, dtype=int)
    rows = [
        {'model': 'constant (train mean)', 'seed': '-', **metrics_for(targets, constant_scores)},
        {'model': 'kc_mean (train, per KC)', 'seed': '-', **metrics_for(targets, kc_scores)}
    ]
    return rows, len(targets), len(test)

def evaluate_run(model_name, seed_dir, data_cfg, test_loader, device):
    summary = json.loads((seed_dir / 'run_summary.json').read_text())
    model = init_model(model_name, summary['model_config'], data_cfg, EMB_TYPE)
    model.load_state_dict(torch.load(seed_dir / f'{EMB_TYPE}_model.ckpt', map_location=device))
    model.eval()

    prediction_dir = EVAL_DIR / 'predictions'
    prediction_dir.mkdir(parents=True, exist_ok=True)
    prediction_path = prediction_dir / f'{model_name}_{seed_dir.name}_test_predictions.txt'
    pykt_auc, _ = evaluate(model, test_loader, model_name, str(prediction_path))
    y_true, y_prob = parse_predictions(prediction_path)
    metrics = metrics_for(y_true, y_prob)
    if abs(metrics['auc'] - pykt_auc) > 1e-6:
        raise RuntimeError(
            f'AUC mismatch for {model_name}/{seed_dir.name}: '
            f'computed {metrics["auc"]} vs pyKT {pykt_auc}'
        )
    return {
        'model': model_name,
        'seed': summary['seed'],
        'n_targets': int(len(y_true)),
        'best_epoch': summary['best_epoch'],
        'valid_auc': summary['valid_auc'],
        **metrics
    }

def markdown_table(frame):
    columns = list(frame.columns)
    lines = ['| ' + ' | '.join(columns) + ' |', '|' + '|'.join('---' for _ in columns) + '|']
    lines.extend('| ' + ' | '.join(str(row[column]) for column in columns) + ' |' for _, row in frame.iterrows())
    return '\n'.join(lines)

data_cfg = load_data_config()
baseline_rows, expected_targets, test_learner_count = baseline_scores(data_cfg)
test_loader = DataLoader(make_dataset('test', data_cfg), batch_size=64, shuffle=False)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

per_run = []
for model_name in MODELS_TO_TRAIN:
    seed_dirs = sorted((ARTIFACTS_DIR / model_name).glob('seed*/'))
    seed_dirs = [directory for directory in seed_dirs if (directory / f'{EMB_TYPE}_model.ckpt').exists()]
    if not seed_dirs:
        print(f'WARNING: no trained checkpoints for {model_name}; run the training cell first')
    for seed_dir in seed_dirs:
        run = evaluate_run(model_name, seed_dir, data_cfg, test_loader, device)
        if run['n_targets'] != expected_targets:
            raise RuntimeError(
                f'{model_name}/{seed_dir.name}: {run["n_targets"]} targets, expected {expected_targets}'
            )
        per_run.append(run)
        print(
            f'{model_name:<5} {seed_dir.name:<8} AUC {run["auc"]:.4f} '
            f'ACC {run["accuracy"]:.4f} LogLoss {run["log_loss"]:.4f}'
        )
if not per_run:
    raise RuntimeError('No trained checkpoints found to evaluate')

runs_df = pd.DataFrame(per_run)
summary_rows = []
for model_name, group in runs_df.groupby('model', sort=False):
    row = {'model': model_name.upper(), 'seeds': len(group)}
    for metric, label in [('auc', 'AUC'), ('accuracy', 'Accuracy'), ('log_loss', 'Log loss')]:
        std = group[metric].std(ddof=1) if len(group) > 1 else float('nan')
        row[label] = f'{group[metric].mean():.4f}' + (f' ± {std:.4f}' if len(group) > 1 else '')
    summary_rows.append(row)
for baseline in baseline_rows:
    summary_rows.append({
        'model': baseline['model'],
        'seeds': '-',
        'AUC': f'{baseline["auc"]:.4f}',
        'Accuracy': f'{baseline["accuracy"]:.4f}',
        'Log loss': f'{baseline["log_loss"]:.4f}'
    })
summary_df = pd.DataFrame(summary_rows)

report_header = (
    '# Test-split comparison (held-out learners only)\n\n'
    f'- Test learners: {test_learner_count}; prediction targets: {expected_targets:,} '
    '(every answer except each learner\'s first)\n'
    '- Model selection used the validation fold only; the test split was not used in training.\n'
    '- Mean ± sample standard deviation over seeds (shown only with multiple seeds).\n'
    '- Synthetic dataset: absolute scores describe the generator, not real learners. '
    'DKT/SAKT use KC ids; AKT also receives item ids.\n'
)
report = (
    report_header + '\n## Summary\n\n' + markdown_table(summary_df)
    + '\n\n## Per run\n\n' + markdown_table(runs_df.round(4)) + '\n'
)
(EVAL_DIR / 'test_comparison.md').write_text(report)
pd.concat([runs_df, pd.DataFrame(baseline_rows)], ignore_index=True).to_csv(EVAL_DIR / 'test_comparison.csv', index=False)
(EVAL_DIR / 'test_comparison.json').write_text(json.dumps({
    'n_test_learners': test_learner_count,
    'n_targets': expected_targets,
    'runs': per_run,
    'baselines': baseline_rows
}, indent=2))

print('\n' + report)
print(f'Saved evaluation reports to {EVAL_DIR}')

Read data from processed file: /content/drive/MyDrive/thesis/data/processed/pykt_english/test_sequences.csv_-1.pkl
file path: /content/drive/MyDrive/thesis/data/processed/pykt_english/test_sequences.csv, qlen: 240, clen: 240, rlen: 240
ts.shape: (14723,), ps.shape: (14723,)
dkt   seed42   AUC 0.8541 ACC 0.7966 LogLoss 0.4403
ts.shape: (14723,), ps.shape: (14723,)
dkt   seed43   AUC 0.8555 ACC 0.7981 LogLoss 0.4389
ts.shape: (14723,), ps.shape: (14723,)
dkt   seed44   AUC 0.8535 ACC 0.7953 LogLoss 0.4418
ts.shape: (14723,), ps.shape: (14723,)
akt   seed42   AUC 0.8564 ACC 0.7977 LogLoss 0.4395
ts.shape: (14723,), ps.shape: (14723,)
akt   seed43   AUC 0.8599 ACC 0.8008 LogLoss 0.4350
ts.shape: (14723,), ps.shape: (14723,)
akt   seed44   AUC 0.8592 ACC 0.8009 LogLoss 0.4358
ts.shape: (14723,), ps.shape: (14723,)
sakt  seed42   AUC 0.8445 ACC 0.7869 LogLoss 0.4550
ts.shape: (14723,), ps.shape: (14723,)
sakt  seed43   AUC 0.8441 ACC 0.7855 LogLoss 0.4575
ts.shape: (14723,), ps.shape: (14723

In [13]:
# ===== HARDENING ANALYSES: stronger baselines, calibration, seed ensemble, paired bootstrap =====
# Reads the prediction files saved by the evaluation cell. Trains nothing and loads no new data
# beyond the train/valid/test CSVs. Run it AFTER the evaluation cell.
import numpy as np
from sklearn.linear_model import LogisticRegression

N_BOOT = 1000            # bootstrap resamples of TEST LEARNERS
BOOT_SEED = 7
M_SMOOTH = 5.0           # shrink item / KC means toward their parent mean (pseudo-count)

data_dir = Path(data_cfg['dpath'])
tv = pd.read_csv(data_dir / data_cfg['train_valid_file'])
te = pd.read_csv(data_dir / data_cfg['test_file'])
train_rows, valid_rows = tv[tv['fold'] == TRAIN_FOLD], tv[tv['fold'] == VALID_FOLD]
qmap = pd.read_csv(PROJECT_ROOT / 'data/processed/question_map.csv').sort_values('questions')
item_kc = qmap['concepts'].to_numpy()
num_q, num_c = data_cfg['num_q'], data_cfg['num_c']
assert len(item_kc) == num_q

def unpadded(row):
    mask = sequence_values(row, 'selectmasks')
    n = sum(1 for v in mask if v != -1)
    return (np.array(sequence_values(row, 'questions')[:n]),
            np.array(sequence_values(row, 'concepts')[:n]),
            np.array(sequence_values(row, 'responses')[:n]))

# ---- statistics from the TRAIN fold only ----
item_sum, item_cnt, kc_sum, kc_cnt = np.zeros(num_q), np.zeros(num_q), np.zeros(num_c), np.zeros(num_c)
for _, row in train_rows.iterrows():
    q, c, r = unpadded(row)
    np.add.at(item_sum, q, r); np.add.at(item_cnt, q, 1)
    np.add.at(kc_sum, c, r);   np.add.at(kc_cnt, c, 1)
global_mean = kc_sum.sum() / kc_cnt.sum()
kc_mean = (kc_sum + M_SMOOTH * global_mean) / (kc_cnt + M_SMOOTH)
item_mean = (item_sum + M_SMOOTH * kc_mean[item_kc]) / (item_cnt + M_SMOOTH)
logit = lambda p: np.log(p / (1 - p))

def target_table(rows):
    """Features for every prediction target (all answers except each learner's first)."""
    y, X, learner = [], [], []
    for li, (_, row) in enumerate(rows.iterrows()):
        q, c, r = unpadded(row)
        csum = np.cumsum(r)
        for t in range(1, len(r)):
            running = (csum[t - 1] + 2 * global_mean) / (t + 2)      # learner's accuracy so far (smoothed)
            X.append([logit(item_mean[q[t]]), logit(kc_mean[c[t]]), logit(running), np.log1p(t)])
            y.append(r[t]); learner.append(li)
    return np.array(y), np.array(X), np.array(learner)

y_valid, X_valid, _ = target_table(valid_rows)
y_test, X_test, learner_test = target_table(te)
assert len(y_test) == expected_targets, (len(y_test), expected_targets)

# Logistic regression is fitted on the VALIDATION learners (item/KC means come from train), so no
# row is used both to build a feature and to fit the model. It knows item difficulty + learner level
# but has no sequence model, so it shows how much DKT/AKT/SAKT add beyond those two effects.
lr = LogisticRegression(C=1.0, max_iter=1000).fit(X_valid, y_valid)

sigmoid = lambda x: 1 / (1 + np.exp(-x))
predictors = {
    'kc_mean (baseline)': sigmoid(X_test[:, 1]),          # column 1 is logit(kc_mean[target KC])
    'item_mean (baseline)': sigmoid(X_test[:, 0]),        # column 0 is logit(item_mean[target item])
    'logreg item+KC+learner (baseline)': lr.predict_proba(X_test)[:, 1],
}

# ---- neural models: load saved per-learner predictions, check alignment, average over seeds ----
pred_dir = EVAL_DIR / 'predictions'
seed_counts, single_run_probs = {}, {}
for model_name in MODELS_TO_TRAIN:
    files = sorted(pred_dir.glob(f'{model_name}_seed*_test_predictions.txt'))
    if not files:
        print(f'WARNING: no prediction files for {model_name}'); continue
    probs = []
    for f in files:
        ts, ps = parse_predictions_by_learner(f)
        assert len(ts) == len(te), f'{f.name}: {len(ts)} lines vs {len(te)} test learners'
        assert np.array_equal(np.concatenate(ts), y_test), f'{f.name}: targets do not line up with test_sequences.csv'
        probs.append(np.concatenate(ps))
    single_run_probs[model_name] = probs
    seed_counts[model_name] = len(files)
    predictors[f'{model_name.upper()} (mean of {len(files)} seed{"s" if len(files) > 1 else ""})'] = np.mean(probs, axis=0)

# ---- metrics, Brier, ECE ----
def calibration(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    rows, ece = [], 0.0
    for b in range(bins):
        m = idx == b
        if m.any():
            rows.append({'bin': f'{edges[b]:.1f}-{edges[b + 1]:.1f}', 'n': int(m.sum()),
                         'mean_predicted': round(float(p[m].mean()), 3), 'observed_rate': round(float(y[m].mean()), 3)})
            ece += m.mean() * abs(p[m].mean() - y[m].mean())
    return pd.DataFrame(rows), float(ece)

metric_rows, reliability = [], {}
for name, p in predictors.items():
    m = metrics_for(y_test, p)
    table, ece = calibration(y_test, np.clip(p, EPS, 1 - EPS))
    reliability[name] = table
    metric_rows.append({'predictor': name, 'AUC': round(m['auc'], 4), 'Accuracy': round(m['accuracy'], 4),
                        'Log loss': round(m['log_loss'], 4), 'Brier': round(float(np.mean((p - y_test) ** 2)), 4),
                        'ECE (10 bins)': round(ece, 4)})
metrics_table = pd.DataFrame(metric_rows)

# ---- paired bootstrap over TEST LEARNERS (resample learners, keep each learner's targets together) ----
idx_by_learner = [np.flatnonzero(learner_test == i) for i in range(len(te))]
rng = np.random.RandomState(BOOT_SEED)
boot = {name: np.empty(N_BOOT) for name in predictors}
for b in range(N_BOOT):
    pick = rng.randint(0, len(te), len(te))
    ix = np.concatenate([idx_by_learner[i] for i in pick])
    yb = y_test[ix]
    for name, p in predictors.items():
        boot[name][b] = roc_auc_score(yb, p[ix])

def label(prefix):
    return next(k for k in predictors if k.startswith(prefix))
auc_ci = pd.DataFrame([{'predictor': n, 'AUC': round(roc_auc_score(y_test, predictors[n]), 4),
                        'CI95_low': round(float(np.percentile(boot[n], 2.5)), 4),
                        'CI95_high': round(float(np.percentile(boot[n], 97.5)), 4)} for n in predictors])

pairs = [('AKT', 'DKT'), ('AKT', 'SAKT'), ('DKT', 'SAKT'),
         ('AKT', 'logreg'), ('DKT', 'logreg'), ('SAKT', 'logreg'), ('AKT', 'item_mean')]
pair_rows = []
for a, b in pairs:
    try:
        na, nb = label(a), label(b)
    except StopIteration:
        continue
    d = boot[na] - boot[nb]
    point = roc_auc_score(y_test, predictors[na]) - roc_auc_score(y_test, predictors[nb])
    pair_rows.append({'A - B': f'{na.split(" (")[0]} - {nb.split(" (")[0]}', 'AUC diff': round(point, 4),
                      'CI95_low': round(float(np.percentile(d, 2.5)), 4), 'CI95_high': round(float(np.percentile(d, 97.5)), 4),
                      'P(A>B)': round(float((d > 0).mean()), 3)})
pairs_table = pd.DataFrame(pair_rows)

report2 = (
    '# Hardening report (test split, saved predictions only)\n\n'
    f'- Predictions are averaged over the trained seeds per model: {seed_counts}\n'
    f'- Bootstrap: {N_BOOT} resamples of the {len(te)} test learners (percentile 95% intervals).\n'
    '  It captures test-learner sampling noise only; it does NOT capture split-to-split variation (one fixed split) '
    'and averaging over seeds hides seed variance (see the per-run table in the main report).\n'
    '- Baselines use TRAIN-fold statistics; the logistic baseline is fitted on VALIDATION learners.\n'
    '- ECE = expected calibration error (lower is better); Brier = mean squared error of the probability.\n\n'
    '## Metrics\n\n' + markdown_table(metrics_table) +
    '\n\n## AUC with 95% bootstrap interval\n\n' + markdown_table(auc_ci) +
    '\n\n## Paired AUC differences (same resampled learners)\n\n' + markdown_table(pairs_table) +
    '\n\n## Reliability tables\n\n' +
    '\n\n'.join(f'**{n}**\n\n' + markdown_table(t) for n, t in reliability.items() if not n.startswith(('kc_mean', 'item_mean'))) + '\n'
)
(EVAL_DIR / 'hardening_report.md').write_text(report2)
metrics_table.to_csv(EVAL_DIR / 'hardening_metrics.csv', index=False)
pairs_table.to_csv(EVAL_DIR / 'hardening_paired_auc.csv', index=False)
print(report2)

# Hardening report (test split, saved predictions only)

- Predictions are averaged over the trained seeds per model: {'dkt': 3, 'akt': 3, 'sakt': 3}
- Bootstrap: 1000 resamples of the 240 test learners (percentile 95% intervals).
  It captures test-learner sampling noise only; it does NOT capture split-to-split variation (one fixed split) and averaging over seeds hides seed variance (see the per-run table in the main report).
- Baselines use TRAIN-fold statistics; the logistic baseline is fitted on VALIDATION learners.
- ECE = expected calibration error (lower is better); Brier = mean squared error of the probability.

## Metrics

| predictor | AUC | Accuracy | Log loss | Brier | ECE (10 bins) |
|---|---|---|---|---|---|
| kc_mean (baseline) | 0.8148 | 0.7702 | 0.488 | 0.1605 | 0.0168 |
| item_mean (baseline) | 0.8063 | 0.764 | 0.5028 | 0.1639 | 0.0218 |
| logreg item+KC+learner (baseline) | 0.8596 | 0.802 | 0.4334 | 0.1399 | 0.0155 |
| DKT (mean of 3 seeds) | 0.8581 | 0.7995 | 0.4354

In [14]:
# Disconnect runtime to save GPU usage
from google.colab import runtime
runtime.unassign()